In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..'); sys.path.append('../../gmsh/')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import json

In [ ]:
h = 5
w = 5
avg_len = 0.1

In [ ]:
import importlib
importlib.reload(periodic_unit_helper)

In [ ]:
from periodic_unit_helper import *

In [ ]:
import pattern_generator_using_gmsh

In [ ]:
sys.path.append('../../gmsh/')

In [ ]:
import mesher_helper

In [ ]:
import pattern_generator_using_gmsh

In [ ]:
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
w = h


In [ ]:
angle = 42
r = 2.5 / np.sqrt(2) * 0.9
r = 2.4

In [ ]:
print("angle", angle, "radius: ", r)
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])

In [ ]:
ipu, m, marker = pattern_generator_using_gmsh.get_three_star(h, 0.2, 0.2, dash_point = dash_point)

In [ ]:
visualization.plot_2d_mesh(m, pointList=marker, width=5, height=5)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= -1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
visualization.plot_2d_mesh(m, pointList=fusedVtx, width=5, height=5)



In [ ]:
np.linspace(0, 3, 61)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 2], 0
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
# fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)

ipu.sheet.pressure = 0.4

In [ ]:
opts.niter = 2000
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, True)

In [ ]:
az_ipu = inflation.InflatableMidSurfacePeriodicUnit(m, fusedVtx, epsilon = 1e-5)
az_ipu.ipu.setVars(ipu.getVars())
az_ipu.ipu.sheet.setUseTensionFieldEnergy(True)
az_ipu.ipu.sheet.setUseHessianProjectedEnergy(False)
az_ipu.ipu.sheet.pressure = ipu.sheet.pressure

In [ ]:

from tri_mesh_viewer import TriMeshViewer
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)

In [ ]:
az_viewer.show()

In [ ]:
def az_cb(it):
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
opts.niter = 400
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = 1e-10, fixedVars = [])
benchmark.report()
min(stiffness_values), max(stiffness_values)